# Exploratory Data Analysis: AI4I 2020 Predictive Maintenance

This notebook is the first exploratory analysis for the predictive maintenance
classification project. The business problem is to identify machines at higher
risk of failure so maintenance actions can be prioritized before costly
breakdowns occur.

The dataset is the public UCI AI4I 2020 Predictive Maintenance dataset. The
objective of this EDA is to understand the schema, target balance, feature
patterns, missingness, duplicates, outliers, and leakage risks before baseline
modeling.

The AI4I dataset is synthetic. Its structure is useful for demonstrating a
professional machine learning workflow, but findings should not be presented as
validated behavior from a real production factory.

## 1. Imports and Setup

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from predictive_maintenance import config
from predictive_maintenance.data import load_dataset
from predictive_maintenance.profiling import profile_dataset

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

FIGURES_DIR = config.FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = config.TARGET_COLUMN
NUMERICAL_FEATURES = list(config.NUMERICAL_FEATURES)
CATEGORICAL_FEATURES = list(config.CATEGORICAL_FEATURES)
LEAKAGE_COLUMNS = list(config.FAILURE_MODE_COLUMNS)

## 2. Load and Validate Data

The dataset is loaded through the project ingestion layer. This keeps path
handling, duplicate-header checks, required-column validation, and deterministic
column ordering centralized in reusable source code.

In [2]:
df = load_dataset()

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(df.head())

display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str)}))

Shape: 10,000 rows x 14 columns


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.100,308.600,1551,42.800,0,0,0,0,0,0,0
1,2,L47181,L,298.200,308.700,1408,46.300,3,0,0,0,0,0,0
2,3,L47182,L,298.100,308.500,1498,49.400,5,0,0,0,0,0,0
3,4,L47183,L,298.200,308.600,1433,39.500,7,0,0,0,0,0,0
4,5,L47184,L,298.200,308.700,1408,40.000,9,0,0,0,0,0,0


,column,dtype
UDI,UDI,int64
Product ID,Product ID,str
Type,Type,str
Air temperature [K],Air temperature [K],float64
Process temperature [K],Process temperature [K],float64
Rotational speed [rpm],Rotational speed [rpm],int64
Torque [Nm],Torque [Nm],float64
Tool wear [min],Tool wear [min],int64
Machine failure,Machine failure,int64
TWF,TWF,int64


## 3. Reusable Profile Summary

In [3]:
profile = profile_dataset(df)

print("Dataset summary")
display(pd.Series(profile["dataset_summary"], name="value"))

print("Missing values")
missing_summary = pd.DataFrame(
    {
        "missing_count": profile["missing_values"]["missing_count_per_column"],
        "missing_percentage": profile["missing_values"][
            "missing_percentage_per_column"
        ],
    }
)
display(missing_summary)

print("Target summary")
display(pd.DataFrame(profile["target_summary"]))

print("ML warnings")
for warning in profile["ml_warnings"]:
    print(f"- {warning}")

print("Schema summary")
for key, value in profile["schema_summary"].items():
    print(f"{key}: {value}")

Dataset summary


row_count                10000
column_count                14
memory_usage_bytes     2010132
duplicate_row_count          0
Name: value, dtype: int64

Missing values


,missing_count,missing_percentage
UDI,0,0.000
Product ID,0,0.000
Type,0,0.000
Air temperature [K],0,0.000
Process temperature [K],0,0.000
Rotational speed [rpm],0,0.000
Torque [Nm],0,0.000
Tool wear [min],0,0.000
Machine failure,0,0.000
TWF,0,0.000


Target summary


,class_counts,class_percentages,imbalance_ratio
0,9661,96.610,28.499
1,339,3.390,28.499


ML warnings
- Highly imbalanced target detected: majority/minority ratio is 28.50.
Schema summary
identifier_columns: ['UDI', 'Product ID']
numerical_features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
categorical_features: ['Type']
target_column: Machine failure
leakage_columns: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']


## 4. Target Distribution

The target is `Machine failure`, where `1` indicates a failure case and `0`
indicates a non-failure case. The failure rate and class imbalance guide the
choice of evaluation metrics and threshold strategy.

In [4]:
target_counts = df[TARGET_COLUMN].value_counts().sort_index()
target_percentages = (target_counts / len(df) * 100).round(3)
target_table = pd.DataFrame(
    {
        "count": target_counts,
        "percentage": target_percentages,
    }
)
target_table.index.name = TARGET_COLUMN

failure_rate = target_percentages.get(1, 0.0)
imbalance_ratio = profile["target_summary"]["imbalance_ratio"]

print(f"Failure rate: {failure_rate:.3f}%")
print(f"Majority/minority imbalance ratio: {imbalance_ratio:.2f}")
display(target_table)

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", ax=ax)
ax.set_title("Machine Failure Target Distribution")
ax.set_xlabel("Machine failure")
ax.set_ylabel("Record count")
ax.set_xticklabels(["No failure", "Failure"], rotation=0)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

Failure rate: 3.390%
Majority/minority imbalance ratio: 28.50


,count,percentage
Machine failure,,
0,9661,96.610
1,339,3.390


/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/4126502156.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Numerical Feature Analysis

The numerical predictive features describe operating conditions and tool wear.
The summaries below review central tendency, spread, and range before any
preprocessing decisions are made.

In [5]:
numerical_summary = df[NUMERICAL_FEATURES].describe().T
numerical_summary["missing_count"] = df[NUMERICAL_FEATURES].isna().sum()
display(numerical_summary)

,count,mean,std,min,25%,50%,75%,max,missing_count
Air temperature [K],10000.000,300.005,2.000,295.300,298.300,300.100,301.500,304.500,0
Process temperature [K],10000.000,310.006,1.484,305.700,308.800,310.100,311.100,313.800,0
Rotational speed [rpm],10000.000,1538.776,179.284,1168.000,1423.000,1503.000,1612.000,2886.000,0
Torque [Nm],10000.000,39.987,9.969,3.800,33.200,40.100,46.800,76.600,0
Tool wear [min],10000.000,107.951,63.654,0.000,53.000,108.000,162.000,253.000,0


In [6]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, feature in zip(axes, NUMERICAL_FEATURES, strict=False):
    ax.hist(df[feature], bins=30)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel("Frequency")

for ax in axes[len(NUMERICAL_FEATURES) :]:
    ax.axis("off")

fig.suptitle("Numerical Feature Distributions", y=1.02)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "numerical_feature_histograms.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/1479446879.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, feature in zip(axes, NUMERICAL_FEATURES, strict=False):
    ax.boxplot(df[feature].dropna(), vert=False)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_yticks([])

for ax in axes[len(NUMERICAL_FEATURES) :]:
    ax.axis("off")

fig.suptitle("Numerical Feature Range and Outlier Review", y=1.02)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "numerical_feature_boxplots.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/3141018742.py:5: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(df[feature].dropna(), vert=False)
/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/3141018742.py:5: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(df[feature].dropna(), vert=False)
/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/3141018742.py:5: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(df[feature].dropna(), vert=False)
/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/3141018742.py:5: MatplotlibDeprecationWarning: vert: bool was deprecated in M

/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/3141018742.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Feature Comparison by Target

The next view compares numerical feature distributions between failure and
non-failure records. Visible differences may be useful for classification, but
this notebook does not make causal claims.

In [8]:
grouped_stats = df.groupby(TARGET_COLUMN)[NUMERICAL_FEATURES].agg(
    ["mean", "median", "std", "min", "max"]
)
display(grouped_stats)

Air temperature [K]                                \
                               mean  median   std     min     max   
Machine failure                                                     
0                           299.974 300.000 1.991 295.300 304.500   
1                           300.886 301.600 2.071 295.600 304.400   

                Process temperature [K]                                \
                                   mean  median   std     min     max   
Machine failure                                                         
0                               309.996 310.000 1.487 305.700 313.800   
1                               310.290 310.400 1.364 306.100 313.700   

                Rotational speed [rpm]                               \
                                  mean   median     std   min   max   
Machine failure                                                       
0                             1540.260 1507.000 167.395  1168  2695   
1                             1496.487 1365.000 384.944  1181  2886   

                Torque [Nm]                             Tool wear [min]  \
                       mean median    std    min    max            mean   
Machine failure                                                           
0                    39.630 39.900  9.472 12.600 70.000         106.694   
1                    50.168 53.700 16.374  3.800 76.600         143.782   

                                         
                 median    std min  max  
Machine failure                          
0               107.000 62.946   0  246  
1               165.000 72.760   0  253

In [9]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, feature in zip(axes, NUMERICAL_FEATURES, strict=False):
    values_by_target = [
        df.loc[df[TARGET_COLUMN] == target_value, feature].dropna()
        for target_value in sorted(df[TARGET_COLUMN].dropna().unique())
    ]
    ax.boxplot(values_by_target, tick_labels=["No failure", "Failure"])
    ax.set_title(feature)
    ax.set_ylabel(feature)

for ax in axes[len(NUMERICAL_FEATURES) :]:
    ax.axis("off")

fig.suptitle("Numerical Features by Target Class", y=1.02)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "numerical_features_by_target.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/1539780950.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Categorical Feature Analysis

In [10]:
type_counts = df["Type"].value_counts().sort_index()
type_percentages = (type_counts / len(df) * 100).round(3)
type_failure_rate = df.groupby("Type")[TARGET_COLUMN].mean().sort_index() * 100

type_summary = pd.DataFrame(
    {
        "count": type_counts,
        "percentage": type_percentages,
        "failure_rate_percentage": type_failure_rate.round(3),
    }
)
display(type_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
type_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Product Type Distribution")
axes[0].set_xlabel("Type")
axes[0].set_ylabel("Record count")
axes[0].tick_params(axis="x", rotation=0)

type_failure_rate.plot(kind="bar", ax=axes[1])
axes[1].set_title("Failure Rate by Product Type")
axes[1].set_xlabel("Type")
axes[1].set_ylabel("Failure rate (%)")
axes[1].tick_params(axis="x", rotation=0)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "type_feature_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

,count,percentage,failure_rate_percentage
Type,,,
H,1003,10.030,2.094
L,6000,60.000,3.917
M,2997,29.970,2.769


/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/692469897.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Correlation Analysis

The correlation matrix uses only numerical predictive features and the binary
target. Identifier columns and failure-mode columns are excluded to avoid
non-predictive signals and target leakage.

In [11]:
correlation_columns = [*NUMERICAL_FEATURES, TARGET_COLUMN]
correlation_matrix = df[correlation_columns].corr(numeric_only=True)
display(correlation_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlation_matrix, vmin=-1, vmax=1)
ax.set_title("Correlation Matrix: Predictive Numerical Features and Target")
ax.set_xticks(np.arange(len(correlation_columns)))
ax.set_yticks(np.arange(len(correlation_columns)))
ax.set_xticklabels(correlation_columns, rotation=45, ha="right")
ax.set_yticklabels(correlation_columns)

for row_index in range(len(correlation_columns)):
    for column_index in range(len(correlation_columns)):
        value = correlation_matrix.iloc[row_index, column_index]
        ax.text(column_index, row_index, f"{value:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
Air temperature [K],1.000,0.876,0.023,-0.014,0.014,0.083
Process temperature [K],0.876,1.000,0.019,-0.014,0.013,0.036
Rotational speed [rpm],0.023,0.019,1.000,-0.875,0.000,-0.044
Torque [Nm],-0.014,-0.014,-0.875,1.000,-0.003,0.191
Tool wear [min],0.014,0.013,0.000,-0.003,1.000,0.105
Machine failure,0.083,0.036,-0.044,0.191,0.105,1.000


/var/folders/9v/n15bf7s130v8rmg6j0zmnyyw0000gn/T/ipykernel_88584/2412673841.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Outlier and Range Review

The observed ranges and box plots help identify unusual operating conditions.
At this stage, values are not removed. In predictive maintenance, extreme
conditions may be informative rather than erroneous.

In [12]:
range_review = pd.DataFrame(
    {
        "min": df[NUMERICAL_FEATURES].min(),
        "q1": df[NUMERICAL_FEATURES].quantile(0.25),
        "median": df[NUMERICAL_FEATURES].median(),
        "q3": df[NUMERICAL_FEATURES].quantile(0.75),
        "max": df[NUMERICAL_FEATURES].max(),
    }
)
range_review["iqr"] = range_review["q3"] - range_review["q1"]
range_review["lower_boxplot_fence"] = range_review["q1"] - 1.5 * range_review["iqr"]
range_review["upper_boxplot_fence"] = range_review["q3"] + 1.5 * range_review["iqr"]
display(range_review)

outlier_counts = {}
for feature in NUMERICAL_FEATURES:
    lower = range_review.loc[feature, "lower_boxplot_fence"]
    upper = range_review.loc[feature, "upper_boxplot_fence"]
    outlier_counts[feature] = int(((df[feature] < lower) | (df[feature] > upper)).sum())

outlier_summary = pd.Series(outlier_counts, name="boxplot_rule_outlier_count")
display(outlier_summary.to_frame())

,min,q1,median,q3,max,iqr,lower_boxplot_fence,upper_boxplot_fence
Air temperature [K],295.300,298.300,300.100,301.500,304.500,3.200,293.500,306.300
Process temperature [K],305.700,308.800,310.100,311.100,313.800,2.300,305.350,314.550
Rotational speed [rpm],1168.000,1423.000,1503.000,1612.000,2886.000,189.000,1139.500,1895.500
Torque [Nm],3.800,33.200,40.100,46.800,76.600,13.600,12.800,67.200
Tool wear [min],0.000,53.000,108.000,162.000,253.000,109.000,-110.500,325.500


,boxplot_rule_outlier_count
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],418
Torque [Nm],69
Tool wear [min],0


## 10. Leakage Review

In [13]:
leakage_review = pd.DataFrame(
    {
        "column": LEAKAGE_COLUMNS,
        "reason_excluded": (
            "Failure-mode indicator derived from the failure outcome; including "
            "it would leak target information into binary failure prediction."
        ),
    }
)
display(leakage_review)

,column,reason_excluded
0,TWF,Failure-mode indicator derived from the failur...
1,HDF,Failure-mode indicator derived from the failur...
2,PWF,Failure-mode indicator derived from the failur...
3,OSF,Failure-mode indicator derived from the failur...
4,RNF,Failure-mode indicator derived from the failur...


The columns `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` must remain excluded from
binary failure model inputs. They are useful for auditing and explanatory data
review, but they are target-derived failure-mode indicators.

## 11. Modeling Implications

In [14]:
modeling_implications = [
    "Use stratified train/validation/test splits because failures are rare.",
    (
        "Evaluate with precision, recall, F1, PR-AUC, and ROC-AUC; "
        "do not rely on accuracy alone."
    ),
    (
        "Tune the operating threshold on validation data based on "
        "maintenance objectives."
    ),
    "Encode the Type categorical feature before baseline modeling.",
    "Scale numerical features for distance-based models and logistic regression.",
    "Exclude UDI, Product ID, and all failure-mode columns from model inputs.",
    (
        "Treat outliers as potentially informative operating conditions "
        "unless proven erroneous."
    ),
]

for item in modeling_implications:
    print(f"- {item}")

- Use stratified train/validation/test splits because failures are rare.
- Evaluate with precision, recall, F1, PR-AUC, and ROC-AUC; do not rely on accuracy alone.
- Tune the operating threshold on validation data based on maintenance objectives.
- Encode the Type categorical feature before baseline modeling.
- Scale numerical features for distance-based models and logistic regression.
- Exclude UDI, Product ID, and all failure-mode columns from model inputs.
- Treat outliers as potentially informative operating conditions unless proven erroneous.


## 12. Final Summary

In [15]:
summary_findings = [
    (
        f"The dataset contains {len(df):,} rows and "
        f"{len(df.columns):,} columns after validation."
    ),
    (
        f"The observed failure rate is {failure_rate:.3f}%, "
        "indicating a strongly imbalanced target."
    ),
    (f"Duplicate row count is {profile['dataset_summary']['duplicate_row_count']:,}."),
    (
        "Total missing cells reported by the profile: "
        f"{profile['missing_values']['total_missing_cells']:,}."
    ),
    (
        "Numerical operating features show different ranges and spreads "
        "that should be reviewed before scaling."
    ),
    (
        "Feature comparisons by target suggest some operating conditions "
        "may help distinguish failures."
    ),
    (
        "The Type feature has a small fixed category set and should be "
        "encoded for modeling."
    ),
    (
        "Correlation analysis excludes identifiers and failure-mode columns "
        "to avoid leakage."
    ),
    (
        "Unusual values are retained for now because they may represent "
        "meaningful operating conditions."
    ),
    (
        "Baseline modeling should prioritize leakage control, stratification, "
        "imbalance-aware metrics, and threshold tuning."
    ),
]

for finding in summary_findings:
    print(f"- {finding}")

- The dataset contains 10,000 rows and 14 columns after validation.
- The observed failure rate is 3.390%, indicating a strongly imbalanced target.
- Duplicate row count is 0.
- Total missing cells reported by the profile: 0.
- Numerical operating features show different ranges and spreads that should be reviewed before scaling.
- Feature comparisons by target suggest some operating conditions may help distinguish failures.
- The Type feature has a small fixed category set and should be encoded for modeling.
- Correlation analysis excludes identifiers and failure-mode columns to avoid leakage.
- Unusual values are retained for now because they may represent meaningful operating conditions.
- Baseline modeling should prioritize leakage control, stratification, imbalance-aware metrics, and threshold tuning.
